In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
evo_chains = spark.sql(f"""
    WITH pokemon_species_variety AS (
        SELECT
            s.nat_dex_pokedex_no
            ,s.nat_dex_pokemon_name
            ,v.name AS pokemon_name
            ,s.id AS species_id
            ,s.is_legendary
            ,s.is_mythical
            ,s.has_gender_differences
            ,s.forms_switchable
            ,s.growth_rate
            ,s.evolution_chain
            ,s.generation
            ,v.id AS variety_id
            ,v.is_default
            ,v.abilities
            ,v.hidden_ability
            ,v.type_1
            ,v.type_2
        FROM 
            {BRONZE_DATABASE_PREFIX}.species s

        LEFT JOIN {BRONZE_DATABASE_PREFIX}.varieties v
        ON s.id = v.species_id
    )
    ,evo_chain_expanded AS (
        SELECT 
            ec.evo_chain_url
            ,ec.evo_chain_id

            ,ec.basic_stage_pokemon_name AS basic_pokemon_name
            ,basic.nat_dex_pokedex_no AS basic_nat_dex_pokedex_no
            ,basic.nat_dex_pokemon_name AS basic_nat_dex_pokemon_name
            ,basic.species_id AS basic_species_id
            ,basic.variety_id AS basic_variety_id

            ,ec.stage_1_pokemon_name
            ,stage_1.nat_dex_pokedex_no AS stage_1_nat_dex_pokedex_no
            ,stage_1.nat_dex_pokemon_name AS stage_1_nat_dex_pokemon_name
            ,stage_1.species_id AS stage_1_species_id
            ,stage_1.variety_id AS stage_1_variety_id

            ,ec.stage_2_pokemon_name
            ,stage_2.nat_dex_pokedex_no AS stage_2_nat_dex_pokedex_no
            ,stage_2.nat_dex_pokemon_name AS stage_2_nat_dex_pokemon_name
            ,stage_2.species_id AS stage_2_species_id
            ,stage_2.variety_id AS stage_2_variety_id
        FROM 
            {BRONZE_DATABASE_PREFIX}.evolution_chain ec

        LEFT JOIN pokemon_species_variety basic
        ON ec.evo_chain_url = basic.evolution_chain
        AND ec.basic_stage_pokemon_name = basic.pokemon_name

        LEFT JOIN pokemon_species_variety stage_1
        ON ec.evo_chain_url = stage_1.evolution_chain
        AND ec.stage_1_pokemon_name = stage_1.pokemon_name

        LEFT JOIN pokemon_species_variety stage_2
        ON ec.evo_chain_url = stage_2.evolution_chain
        AND ec.stage_2_pokemon_name = stage_2.pokemon_name
    )
    SELECT 
        evo_chain_id
        ,basic_nat_dex_pokedex_no AS nat_dex_pokedex_no
        ,basic_species_id AS species_id
        ,basic_variety_id AS variety_id
        ,basic_pokemon_name AS pokemon_name
        ,NULL AS evolves_from_variety_id
        ,NULL AS evolves_from_pokemon_name
        ,stage_1_variety_id AS evolves_to_variety_id
        ,stage_1_pokemon_name AS evolves_to_pokemon_name
    FROM 
        evo_chain_expanded
    WHERE 
        basic_variety_id IS NOT NULL

    UNION 

    SELECT 
        evo_chain_id
        ,stage_1_nat_dex_pokedex_no AS nat_dex_pokedex_no
        ,stage_1_species_id AS species_id
        ,stage_1_variety_id AS variety_id
        ,stage_1_pokemon_name AS pokemon_name
        ,basic_variety_id AS evolves_from_variety_id
        ,basic_pokemon_name AS evolves_from_pokemon_name
        ,stage_2_variety_id AS evolves_to_variety_id
        ,stage_2_pokemon_name AS evolves_to_pokemon_name
    FROM 
        evo_chain_expanded
    WHERE 
        stage_1_variety_id IS NOT NULL

    UNION 

    SELECT 
        evo_chain_id
        ,stage_2_nat_dex_pokedex_no AS nat_dex_pokedex_no
        ,stage_2_species_id AS species_id
        ,stage_2_variety_id AS variety_id
        ,stage_2_pokemon_name AS pokemon_name
        ,stage_1_variety_id AS evolves_from_variety_id
        ,stage_1_pokemon_name AS evolves_from_pokemon_name
        ,NULL AS evolves_to_variety_id
        ,NULL AS evolves_to_pokemon_name
    FROM 
        evo_chain_expanded
    WHERE 
        stage_2_variety_id IS NOT NULL
""")

evo_chains.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{SILVER_DATABASE_PREFIX}.evolution_chains")

In [0]:
combined = spark.sql(f"""
    SELECT 
        s.nat_dex_pokedex_no
        ,s.nat_dex_pokemon_name
        ,s.id AS species_id
        ,s.name AS species_name
        ,s.capture_rate
        ,s.base_happiness
        ,s.is_baby
        ,s.is_legendary
        ,s.is_mythical
        ,s.hatch_counter
        ,s.has_gender_differences
        ,s.forms_switchable
        ,s.growth_rate
        ,s.generation
        ,v.id AS variety_id
        ,v.name AS variety_pokemon_name
        ,v.is_default
        ,v.abilities
        ,v.hidden_ability
        ,v.hp
        ,v.attack
        ,v.defence
        ,v.special_attack
        ,v.special_defence
        ,v.speed
        ,v.type_1
        ,v.type_2
        ,f.id AS form_id
        ,f.name AS form_pokemon_name
        ,f.form_name
        ,f.form_order
        ,f.is_mega
        ,f.is_battle_only
        ,f.trigger_conditions
    FROM 
        {BRONZE_DATABASE_PREFIX}.species s

    LEFT JOIN {BRONZE_DATABASE_PREFIX}.varieties v
    ON s.id = v.species_id

    LEFT JOIN {BRONZE_DATABASE_PREFIX}.forms f
    ON v.id = f.variety_id
""")

combined.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{SILVER_DATABASE_PREFIX}.combined_species_varieties_forms")